# Momentum Transformer Masterclass: Deep Dive

This notebook provides a live walkthrough of the **Momentum Transformer** architecture and its application to a global macro futures portfolio. We will cover:
1. **Feature Engineering**: Multi-scale MACD and Volatility Scaling.
2. **Variable Selection Network (VSN)**: How the model prioritizes assets.
3. **Attention Maps**: Visualizing temporal dependencies and market regimes.
4. **Cross-Sectional Backtest**: Performance evaluation with realistic transaction costs.

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from models.architecture import MomentumTransformer
from research.final_backtest_cross_sectional import run_backtest_multi

# Setup aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
os.makedirs("research", exist_ok=True)

## 1. Feature Engineering: Multi-Scale Momentum
We use MACD-style signals at multiple scales [10, 21, 63, 126, 252] days to capture both fast and slow trends.

In [ ]:
df = pd.read_parquet('data/processed/futures_1d.parquet')
symbol = 'ES.c.0' # S&P 500
asset_df = df[df['symbol'] == symbol].last('2Y')

plt.figure(figsize=(14, 6))
for w in [10, 63, 252]:
    plt.plot(asset_df.index, asset_df[f'macd_{w}'], label=f'MACD-{w}')

plt.title(f"Multi-Scale Momentum Signals ({symbol})")
plt.legend()
plt.show()

## 2. Model Interpretability: The Variable Selection Network (VSN)
The VSN weights different features and assets dynamically. Let's load the trained model and see what it's looking at.

In [ ]:
feat_cols = [f'macd_{w}' for w in [10, 21, 63, 126, 252]]
num_assets = len(df['symbol'].unique())
model = MomentumTransformer(input_dim=len(feat_cols), num_vars=num_assets, hidden_dim=64, num_heads=4, output_dim=num_assets)
model.load_state_dict(torch.load('weights/model_macro_cs_15.pt', map_location='cpu'))
model.eval()
print("Model Loaded Successfully.")

## 3. Attention Maps: Market Memory
We can visualize the self-attention weights to see how the query at the current time step looks back at past market regimes.

In [ ]:
# Prepare a sample window
pivoted = df.pivot_table(index=df.index, columns='symbol', values=feat_cols).sort_index().ffill().bfill()
x_vals = np.stack([pivoted[f].values for f in feat_cols], axis=-1)
sample_x = torch.tensor(x_vals[-64:], dtype=torch.float32).unsqueeze(0)

with torch.no_grad():
    _, attn_weights = model(sample_x, return_attention=True)

weights = attn_weights[0, 0].cpu().numpy()
plt.figure(figsize=(10, 8))
sns.heatmap(weights, cmap='magma')
plt.title("Transformer Temporal Attention (Last 64 Days)")
plt.xlabel("Past Time")
plt.ylabel("Current Time")
plt.show()

## 4. Final Portfolio Performance
Running the cross-sectional backtest with **Abdi-Ranaldo dynamic spreads** and **20bps slippage**.

In [ ]:
print("--- Running Cross-Sectional Backtest ---")
results = run_backtest_multi(model, df, feat_cols, ann_factor=252, base_cost=0.002)

plt.figure(figsize=(12, 6))
plt.plot(results['dates'], results['cum_pnl'], label=f"Net PnL (Sharpe: {results['sr']:.2f})")
plt.axvline(pd.to_datetime('2025-01-01'), color='red', linestyle='--', label='Out-of-Sample')
plt.title("Cross-Sectional Portfolio: Global Macro Futures")
plt.ylabel("Cumulative Net Return")
plt.legend()
plt.show()

print(f"Final Portfolio Sharpe: {results['sr']:.2f}")
print(f"Annualized Turnover: {results['turnover']:.1f}x")